In [0]:
# Instead of os.environ.get(), use dbutils.secrets.get()
from pyspark.sql.dataframe import DataFrame
from pyspark.sql.functions import xxhash64, col, datediff

def create_connection():
    return spark.read.format("jdbc") \
    .option("url", f"jdbc:mysql://{dbutils.secrets.get('wheelie', 'MYSQL_HOST')}/{dbutils.secrets.get('wheelie', 'MYSQL_DB')}") \
    .option("user", dbutils.secrets.get('wheelie', 'MYSQL_USERNAME')) \
    .option("password", dbutils.secrets.get('wheelie', 'MYSQL_PASSWORD'))

# mode "overwrite" or "append"
# https://stackoverflow.com/questions/66340758/insert-or-update-a-delta-table-from-a-dataframe-in-pyspark
def write_dim(df: DataFrame, table_name: str):
    display(df)
    df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"wheelie.data_warehouse.{table_name}")

c = create_connection()




In [0]:
service_df =  c.option("dbtable", "service").load()

fact_service = service_df.select(
    "service_id",
    "service_date",
    "service_type",
    "service_cost",
    "inventory_id",
).withColumn(
    "service_key", xxhash64("service_id"),
).withColumn(
    "service_date_key", xxhash64("service_date"),
).withColumn(
    "car_key", xxhash64("inventory_id")
).drop(
    "service_date",
    "inventory_id",
)

write_dim(fact_service, "fact_service")


In [0]:
staff_df = c.option("dbtable", "staff").load()
inventory_df = c.option("dbtable", "inventory").load()
payment_df = c.option("dbtable", "payment").load()

staff_df = staff_df.select(
    col("staff_id").alias("staff_staff_id"),
    "store_id"
)
payment_df = payment_df.select(
    col("rental_id").alias("payment_rental_id"),
    "payment_date",
    col("amount").alias("payment_amount")
)

In [0]:
rental_dataframe = c.option("dbtable", "rental").load()

rental_with_staff = rental_dataframe.join(
    staff_df, rental_dataframe.staff_id == staff_df.staff_staff_id, "left"
)

rental_with_staff_inventory = rental_with_staff.join(
    inventory_df.select("inventory_id", "car_id"),
    rental_with_staff.inventory_id == inventory_df.inventory_id,
    "left",
)

rental_with_staff_inventory_payment = rental_with_staff_inventory.join(
    payment_df,
    rental_with_staff_inventory.rental_id == payment_df.payment_rental_id,
    "left",
)

fact_rental = (
    rental_with_staff_inventory_payment.select(
        "rental_id",
        "rental_rate",
        "payment_amount",
        "customer_id",
        "car_id",
        "staff_staff_id",
        "store_id",
        "rental_date",
        "return_date",
        "payment_date",
        "payment_deadline"
    )
    .withColumn("rental_key", xxhash64(col("rental_id")))
    .withColumn("customer_key", xxhash64(col("customer_id")))
    .withColumn("car_key", xxhash64(col("car_id")))
    .withColumn("staff_key", xxhash64(col("staff_staff_id")))
    .withColumn("store_key", xxhash64(col("store_id")))
    .withColumn("rental_date_key", xxhash64(col("rental_date")))
    .withColumn("return_date_key", xxhash64(col("return_date")))
    .withColumn("payment_date_key", xxhash64(col("payment_date")))
    .withColumn("payment_deadline_date_key", xxhash64(col("payment_deadline")))
    .withColumn(
        "rental_amount",
        col("rental_rate") * datediff(col("return_date"), col("rental_date")),
    )
    .withColumn("rental_duration", datediff(col("return_date"), col("rental_date")))
    .withColumn(
        "payment_delay_duration", datediff(col("payment_date"), col("payment_deadline"))
    )
    .drop(
        "customer_id",
        "car_id",
        "staff_staff_id",
        "store_id",
        "rental_date",
        "return_date",
        "payment_date",
        "payment_deadline"
    )
)

write_dim(fact_rental, "fact_rental")